# Tax Compliance AutoML: Risk-Based Auditing on Digital Economy

**Author:** Izam Rosiawan (Telkom University Surabaya) & Sulthan  
**Study Context:** Compliance Risk Management (CRM) for Indonesian Digital Commerce (BPS Indicators Integration)  
**Standard:** SINTA 2 / Scopus Open-Science Reproducible Research (`seed=42`)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.samplers import TPESampler

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc, 
    f1_score, precision_score, recall_score, roc_curve, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
np.random.seed(SEED)
print('All statistical and machine learning dependencies loaded successfully.')

## 1. Data Ingestion & Exploratory Data Analysis (EDA)

Dataset mengintegrasikan data statistik e-commerce regional Badan Pusat Statistik (BPS) dengan fitur transaksi perbankan (*payment gateway*), logistik, serta data Surat Pemberitahuan (SPT) fiskal.

In [ ]:
data_path = os.path.join('data', 'bps_e_commerce_tax_compliance.csv')
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape}')
display(df.head())
print('\nClass Balance Distribution:')
print(df['target_compliance_risk'].value_counts(normalize=True))

## 2. Anti-Leakage Train-Test Split & Feature Preprocessing

Pemisahan dataset 80:20 dilakukan terlebih dahulu sebelum scaling untuk mencegah kebocoran data (*data leakage*).

In [ ]:
X = df.drop(columns=['provinsi', 'target_compliance_risk'])
y = df['target_compliance_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}')

## 3. Baseline Modeling: Logistic Regression & Random Forest

In [ ]:
# Logistic Regression
lr = LogisticRegression(random_state=SEED, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]
lr_preds = lr.predict(X_test_scaled)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEED)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]
rf_preds = rf.predict(X_test)

print(f'Logistic Regression ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}')
print(f'Random Forest ROC-AUC:       {roc_auc_score(y_test, rf_probs):.4f}')

## 4. Automated Machine Learning (AutoML) with Optuna Bayesian Search

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def objective(trial):
    classifier_name = trial.suggest_categorical('classifier', ['LightGBM', 'XGBoost'])
    
    if classifier_name == 'LightGBM':
        params = {
            'n_estimators': trial.suggest_int('lgb_n_estimators', 50, 250),
            'learning_rate': trial.suggest_float('lgb_lr', 0.01, 0.2, log=True),
            'num_leaves': trial.suggest_int('lgb_num_leaves', 15, 127),
            'max_depth': trial.suggest_int('lgb_max_depth', 3, 10),
            'subsample': trial.suggest_float('lgb_subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('lgb_colsample', 0.6, 1.0),
            'random_state': SEED,
            'verbose': -1
        }
        model = lgb.LGBMClassifier(**params)
    else:
        params = {
            'n_estimators': trial.suggest_int('xgb_n_estimators', 50, 250),
            'learning_rate': trial.suggest_float('xgb_lr', 0.01, 0.2, log=True),
            'max_depth': trial.suggest_int('xgb_max_depth', 3, 10),
            'subsample': trial.suggest_float('xgb_subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('xgb_colsample', 0.6, 1.0),
            'random_state': SEED,
            'eval_metric': 'logloss'
        }
        model = xgb.XGBClassifier(**params)
        
    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_tr, y_tr)
        preds = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))
    return np.mean(scores)

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
study.optimize(objective, n_trials=25)

best_params = study.best_params
print('Best Model Hyperparameters:', best_params)

if best_params['classifier'] == 'LightGBM':
    best_model = lgb.LGBMClassifier(
        n_estimators=best_params['lgb_n_estimators'],
        learning_rate=best_params['lgb_lr'],
        num_leaves=best_params['lgb_num_leaves'],
        max_depth=best_params['lgb_max_depth'],
        subsample=best_params['lgb_subsample'],
        colsample_bytree=best_params['lgb_colsample'],
        random_state=SEED,
        verbose=-1
    )
else:
    best_model = xgb.XGBClassifier(
        n_estimators=best_params['xgb_n_estimators'],
        learning_rate=best_params['xgb_lr'],
        max_depth=best_params['xgb_max_depth'],
        subsample=best_params['xgb_subsample'],
        colsample_bytree=best_params['xgb_colsample'],
        random_state=SEED,
        eval_metric='logloss'
    )

best_model.fit(X_train, y_train)
automl_probs = best_model.predict_proba(X_test)[:, 1]
automl_preds = best_model.predict(X_test)
print(f'AutoML Best Test ROC-AUC: {roc_auc_score(y_test, automl_probs):.4f}')

## 5. Comparative Evaluation & Publication Visualizations (300 DPI)

Menghasilkan kurva ROC dan kurva *Cumulative Decile Lift* untuk analisis efisiensi audit perpajakan.

In [ ]:
plt.figure(figsize=(7, 5), dpi=300)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)
fpr_am, tpr_am, _ = roc_curve(y_test, automl_probs)

plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, lr_probs):.4f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, rf_probs):.4f})')
plt.plot(fpr_am, tpr_am, label=f'AutoML XGBoost (AUC = {roc_auc_score(y_test, automl_probs):.4f})', linewidth=2, color='darkblue')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves: Comparative Tax Risk Classification')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join('images', 'figure1_roc_auc_curve.png'))
plt.show()

# Decile Lift Calculation
gains_df = pd.DataFrame({'y_true': y_test, 'prob': automl_probs})
gains_df['decile'] = pd.qcut(gains_df['prob'], q=10, labels=False, duplicates='drop')
gains_df['decile'] = 10 - gains_df['decile']

decile_summary = gains_df.groupby('decile')['y_true'].sum().reset_index()
decile_summary['cum_gains'] = decile_summary['y_true'].cumsum()
decile_summary['cum_gains_pct'] = decile_summary['cum_gains'] / decile_summary['y_true'].sum() * 100.0

plt.figure(figsize=(7, 5), dpi=300)
plt.plot(decile_summary['decile'], decile_summary['cum_gains_pct'], marker='o', color='#b91c1c', linewidth=2, label='AutoML Cumulative Audit Yield')
plt.plot([1, 10], [10, 100], 'k--', alpha=0.4, label='Random Audit Baseline')
plt.xlabel('Risk Decile (1 = Highest Risk, 10 = Lowest)')
plt.ylabel('Cumulative Identified Non-Compliant Entities (%)')
plt.title('Cumulative Audit Yield by Risk Decile')
plt.xticks(range(1, 11))
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join('images', 'figure2_cumulative_gains_decile.png'))
plt.show()

print(f'Top 20% Risk Decile Captures: {decile_summary.loc[decile_summary["decile"] <= 2, "cum_gains_pct"].max():.2f}% of all high-risk taxpayers.')